# Decoder benchmark — how well does Decode & Infer recover stated affect?

Implements §12.5 of *ASA Design v0.4*. **This is a research measure, not a test.** A table
scoring 60% is a finding; nothing here should ever gate the build (§15).

It runs today because the decoder is *stateless* — no agent, no model, no clock, no robot.
That tractability is the second reason §8.1 gives for statelessness, and this is it being
cashed in.

## What this can and cannot measure

- **Can:** five-way dominant-axis agreement (four axes plus neutral), and which sentences fail
  and why, from the `rationale` the decoder writes.
- **Cannot yet:** per-axis intensity error, because the labelled set records one intended *axis*
  per sentence rather than a full vector. §12.5 wants both. That gap is part of **open question
  16**, along with which measure is the headline figure and how a tie on the dominant axis
  resolves — a tie this set deliberately contains.

**Never aggregate across axes into one error number.** `basic4/1` and `ekman6/1` both declare
`metric=False`: a mean, and still more a Euclidean distance, asserts a commensurability the
representation does not have. The `metric` flag exists to be branched on.


In [1]:
import csv
from pathlib import Path

from asa.core.affect import AffectVector, Utterance
from asa.core.representations import BASIC4, EKMAN6
from asa.perception.decode_keyword import (
    BASIC4_KEYWORDS,
    EKMAN6_KEYWORDS,
    KeywordDecoder,
)

BENCHMARK = Path("data_in/decoder_benchmark.csv")   # relative to the repo root
NEUTRAL_THRESHOLD = 0.15                            # [affect] neutral_threshold, §11

rows = list(csv.DictReader(BENCHMARK.open(encoding="utf-8")))
print(f"{len(rows)} labelled sentences")
rows[:3]


20 labelled sentences


[{'text': 'I am so happy', 'basic4/1': 'happiness', 'ekman6/1': 'happiness'},
 {'text': "I'm absolutely delighted",
  'basic4/1': 'happiness',
  'ekman6/1': 'happiness'},
 {'text': 'I was gutted', 'basic4/1': 'sadness', 'ekman6/1': 'sadness'}]

## One set of sentences, one label column per representation

The sentences are **shared** and the labels are **not**. Each representation defines its own
ground truth (§12.6), so `anger_disgust` is meaningless to `ekman6/1` and `disgust` is
meaningless to `basic4/1` — but two *separate sentence lists* would drift, and a comparison
would then measure the drift rather than the representations. That is precisely the trap the two
keyword lexicons carry (§8.4); one file with two label columns makes it unreachable here.

An empty label means *neutral*. A representation that genuinely could not express a sentence's
affect would need a third marker — neither of these two does, since `basic4/1`'s merged axes span
everything `ekman6/1` splits.


In [2]:
def dominant(vector: AffectVector, threshold: float = NEUTRAL_THRESHOLD) -> str | None:
    """The five-way quantiser: the strongest axis, or None for neutral."""
    top = max(vector.values, key=lambda axis: vector.values[axis])
    return str(top) if vector.values[top] > threshold else None


def tied_axes(vector: AffectVector, threshold: float = NEUTRAL_THRESHOLD) -> list[str]:
    """Every axis sharing the top magnitude — open question 16's unresolved case."""
    best = max(vector.values.values())
    if best <= threshold:
        return []
    return [str(a) for a, v in vector.values.items() if v == best]


async def run(representation, table, column):
    """Decode every row and record what happened."""
    decoder = KeywordDecoder(representation, table)
    out = []
    for row in rows:
        intended = row[column] or None
        utterance = Utterance(
            text=row["text"],
            source="input:benchmark",
            intended=AffectVector(representation.id, {intended: 1.0}) if intended else None,
        )
        observation = await decoder.decode(utterance)
        out.append({
            "text": row["text"],
            "intended": intended,
            "decoded": dominant(observation.affect),
            "tied": tied_axes(observation.affect),
            "rationale": observation.rationale,
        })
    return out


## Score each representation against its own labels


In [3]:
basic4 = await run(BASIC4, BASIC4_KEYWORDS, "basic4/1")
ekman6 = await run(EKMAN6, EKMAN6_KEYWORDS, "ekman6/1")

for name, result in [("basic4/1", basic4), ("ekman6/1", ekman6)]:
    hits = sum(r["decoded"] == r["intended"] for r in result)
    print(f"{name}  dominant-axis agreement: {hits}/{len(result)} ({hits / len(result):.0%})")


basic4/1  dominant-axis agreement: 16/20 (80%)
ekman6/1  dominant-axis agreement: 16/20 (80%)


**Read those two numbers with care.** They are *within-representation* accuracies, and comparing
them ranks the decoders-with-their-tables rather than the theories — each was scored against its
own ground truth (§12.6). Ranking representations needs a common downstream criterion, and
§12.3's recognition accuracy is the one this project has.

## The misses are the point


In [4]:
def show_misses(result, name):
    print(f"=== {name} ===")
    for r in result:
        if r["decoded"] != r["intended"]:
            print(f"  {r['text']!r}")
            print(f"      intended={r['intended']}  decoded={r['decoded']}")
            print(f"      {r['rationale']}")


show_misses(basic4, "basic4/1")
show_misses(ekman6, "ekman6/1")


=== basic4/1 ===
  'I just got the job!'
      intended=happiness  decoded=None
      no keyword matched
  'my dog died last night'
      intended=sadness  decoded=None
      no keyword matched
  'the deadline moved forward by a week'
      intended=fear_surprise  decoded=None
      no keyword matched
  'I am not happy about this'
      intended=sadness  decoded=happiness
      matched: happiness=happy
=== ekman6/1 ===
  'I just got the job!'
      intended=happiness  decoded=None
      no keyword matched
  'my dog died last night'
      intended=sadness  decoded=None
      no keyword matched
  'the deadline moved forward by a week'
      intended=fear  decoded=None
      no keyword matched
  'I am not happy about this'
      intended=sadness  decoded=happiness
      matched: happiness=happy


Expect three kinds of failure, and only one is a defect:

1. **Inference, not decoding.** *"I just got the job!"*, *"my dog died last night"* — affect must
   be inferred from a situation, and a keyword table only decodes affect that was *stated*. This
   is the gap the LLM decoder exists to fill, and it is §19.1's worked scenario failing on
   purpose.
2. **Negation.** *"I am not happy about this"* decodes as happiness. Handling it needs parsing.
3. **A genuine lexicon gap** — a stated-affect sentence whose word is simply missing. *That* is
   the only one worth fixing, and the fix is a table entry.

`rationale` is what tells them apart: `no keyword matched` on a sentence that plainly states an
emotion is a gap, whereas the same on *"my dog died last night"* is the design working as
intended.

Adding a word means deciding **deliberately** which axis it takes in *both* tables. They are
independent by design, and a word in one and not the other produces a difference that reads as
representational and is not (§8.4).

## Ties


In [5]:
for name, result in [("basic4/1", basic4), ("ekman6/1", ekman6)]:
    for r in result:
        if len(r["tied"]) > 1:
            print(f"{name}  {r['text']!r}")
            print(f"          tied on {r['tied']} — {r['rationale']}")


basic4/1  "I'm delighted but astonished"
          tied on ['happiness', 'fear_surprise'] — matched: happiness=delighted, fear_surprise=astonished
ekman6/1  "I'm delighted but astonished"
          tied on ['happiness', 'surprise'] — matched: happiness=delighted, surprise=astonished


A tie means the quantiser has no defined answer, and `dominant()` above resolves it by mapping
order — arbitrary, and therefore **not** a decision this notebook should be making silently. It
is open question 16, and it wants ruling before any result is reported.

It is not an artificial case either: *"I'm delighted but astonished"* is an ordinary sentence,
and the representation holds independent intensities precisely so both axes *can* be high (§5.3).
The forced choice is imposed by the categorical measure, not by the data.

---

A `pandas` DataFrame would present the misses better than the loop above. It is not a dependency
— scaffold decision 6 has them arrive when real code needs one — so adding it to the `notebook`
group is a deliberate choice rather than something this notebook should assume.
